# 02 — Tacotron 2: End-to-End Neural TTS

**Paper:** *Natural TTS Synthesis by Conditioning WaveNet on Mel Spectrogram Predictions* (Shen et al., Google 2018)  
**arXiv:** https://arxiv.org/abs/1712.05884

---

## The TTS Pipeline

Before Tacotron, TTS systems required complex **multi-stage pipelines**:

```
Text
 -> Linguistic analysis (POS, phoneme rules)
 -> Duration model
 -> Acoustic model (F0, spectral envelope)
 -> Vocoder
 -> Audio
```

Each stage required expert knowledge and separate training.

**Tacotron 2** collapses this to **two trainable components:**

```
Text (characters or phonemes)
         |
    [Tacotron 2]    <- seq2seq with attention
         |
  Mel Spectrogram
         |
    [WaveNet / HiFi-GAN vocoder]
         |
      Raw Audio
```

![TTS Pipeline](./figures/tts_pipeline.png)

## Tacotron 2 Architecture

![Tacotron 2 Architecture](./figures/tacotron2_arch.png)

### Encoder
- Character/phoneme embedding (512-dim)
- 3 x Conv1D (512 channels, kernel=5, BatchNorm + ReLU)
- Bidirectional LSTM (256 units each direction -> 512-dim output)

### Attention (Location-Sensitive)
- Extends Bahdanau attention with **location features** — uses previous attention weights as input
- Prevents the decoder from getting stuck on one position
- Output: **context vector** aligning encoder output to current decoder step

### Decoder
- Pre-net: 2 x FC(256) with ReLU + Dropout (always active — adds variation)
- Attention RNN: LSTM(1024)
- Decoder RNN: LSTM(1024)
- Linear projection -> 80 mel bands per frame
- Stop token: sigmoid -> whether to stop generation

### Post-net
- 5 x Conv1D to refine the mel spectrogram output (residual addition)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# ── Tacotron 2 Components ──

class ConvBNReLU(nn.Module):
    def __init__(self, channels, kernel_size=5, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
    def forward(self, x): return self.net(x)


class Encoder(nn.Module):
    def __init__(self, vocab_size=148, embed_dim=512, n_convs=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.Sequential(*[ConvBNReLU(embed_dim) for _ in range(n_convs)])
        self.lstm  = nn.LSTM(embed_dim, embed_dim // 2, batch_first=True, bidirectional=True)

    def forward(self, x, lengths=None):
        # x: (B, T_text) character indices
        x = self.embedding(x).transpose(1, 2)   # (B, embed, T)
        x = self.convs(x).transpose(1, 2)        # (B, T, embed)
        out, _ = self.lstm(x)
        return out   # (B, T_text, 512)


class Prenet(nn.Module):
    def __init__(self, in_dim=80, sizes=(256, 256)):
        super().__init__()
        self.layers = nn.ModuleList()
        prev = in_dim
        for s in sizes:
            self.layers.append(nn.Linear(prev, s))
            prev = s

    def forward(self, x):
        # Dropout always active (adds stochasticity for variation)
        for fc in self.layers:
            x = F.dropout(F.relu(fc(x)), p=0.5, training=True)
        return x


class LocationAttention(nn.Module):
    def __init__(self, attn_dim=128, n_filters=32, kernel=31):
        super().__init__()
        self.query_layer   = nn.Linear(1024, attn_dim, bias=False)
        self.memory_layer  = nn.Linear(512, attn_dim, bias=False)
        self.location_conv = nn.Conv1d(2, n_filters, kernel, padding=kernel//2, bias=False)
        self.location_dense= nn.Linear(n_filters, attn_dim, bias=False)
        self.v             = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, query, memory, attn_weights_cat):
        # query:           (B, 1024)
        # memory:          (B, T_enc, 512)
        # attn_weights_cat: (B, 2, T_enc)  [previous + cumulative attention]
        q   = self.query_layer(query).unsqueeze(1)           # (B, 1, attn_dim)
        m   = self.memory_layer(memory)                       # (B, T_enc, attn_dim)
        loc = self.location_dense(
                  self.location_conv(attn_weights_cat).transpose(1, 2)
              )                                               # (B, T_enc, attn_dim)
        energy = self.v(torch.tanh(q + m + loc)).squeeze(-1) # (B, T_enc)
        return F.softmax(energy, dim=-1)                      # (B, T_enc)


class Postnet(nn.Module):
    def __init__(self, n_mels=80, n_convs=5, channels=512, kernel=5):
        super().__init__()
        layers = []
        for i in range(n_convs):
            in_ch  = n_mels if i == 0 else channels
            out_ch = n_mels if i == n_convs - 1 else channels
            act    = nn.Identity() if i == n_convs - 1 else nn.Tanh()
            layers += [nn.Conv1d(in_ch, out_ch, kernel, padding=kernel//2),
                       nn.BatchNorm1d(out_ch), act, nn.Dropout(0.5)]
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)   # residual added outside


# Quick shape tests
enc = Encoder(vocab_size=100, embed_dim=512)
pre = Prenet(in_dim=80)
pst = Postnet(n_mels=80)

text_ids = torch.randint(0, 100, (2, 20))
enc_out  = enc(text_ids)
print(f"Encoder output:  {enc_out.shape}   (B, T_text, 512)")

mel_frame = torch.randn(2, 80)
pre_out   = pre(mel_frame)
print(f"Prenet output:   {pre_out.shape}   (B, 256)")

mel_seq   = torch.randn(2, 80, 100)
pst_out   = pst(mel_seq)
print(f"Postnet output:  {pst_out.shape}   (B, 80, T_mel)")

In [ ]:
# ── Tacotron 2 Training Loss ──

class Tacotron2Loss(nn.Module):
    def forward(self, mel_pred, mel_pred_postnet, stop_pred, mel_target, stop_target):
        # mel_pred:         (B, n_mels, T)  before postnet
        # mel_pred_postnet: (B, n_mels, T)  after postnet
        # stop_pred:        (B, T)          stop token logits
        # mel_target:       (B, n_mels, T)
        # stop_target:      (B, T)          binary

        mel_loss_1 = F.mse_loss(mel_pred, mel_target)
        mel_loss_2 = F.mse_loss(mel_pred_postnet, mel_target)
        stop_loss  = F.binary_cross_entropy_with_logits(stop_pred, stop_target)
        return mel_loss_1 + mel_loss_2 + stop_loss, mel_loss_1.item(), stop_loss.item()


criterion = Tacotron2Loss()

B, T_mel, n_mels = 2, 100, 80
mel_pred         = torch.randn(B, n_mels, T_mel)
mel_pred_postnet = torch.randn(B, n_mels, T_mel)
stop_pred        = torch.randn(B, T_mel)
mel_target       = torch.randn(B, n_mels, T_mel)
stop_target      = torch.zeros(B, T_mel)
stop_target[:, -5:] = 1.0   # last 5 frames are "stop"

loss, mel_l, stop_l = criterion(mel_pred, mel_pred_postnet, stop_pred, mel_target, stop_target)
print(f"Total loss:  {loss.item():.4f}")
print(f"Mel loss:    {mel_l:.4f}")
print(f"Stop loss:   {stop_l:.4f}")

## Attention: How Tacotron 2 Aligns Text to Speech

![Tacotron 2 Attention](./figures/tacotron2_attention.png)

The attention matrix shows which **text position** the decoder is focusing on at each **audio frame**.  
A well-trained model produces a clean diagonal — it reads left to right through the text.

**Location-sensitive attention** adds the previous attention weights and their cumulative sum  
as extra features — this prevents the decoder from jumping around or repeating words.

**What can go wrong:**
- Attention collapses to one position (skip words)
- Attention oscillates (repeat words or skip)
- Attention goes backwards (speech out of order)

These issues were common with vanilla Bahdanau attention — location sensitivity largely fixes them.

In [ ]:
# ── Attention Alignment Visualization ──
# Simulate a clean diagonal attention matrix (what a trained model produces)
import matplotlib.ticker as ticker

T_text = 30   # text length (characters)
T_mel  = 200  # mel frames

# Create a smooth diagonal attention pattern
def simulate_attention(T_text, T_mel, noise=0.05):
    attn = np.zeros((T_mel, T_text))
    for t in range(T_mel):
        center = int(t / T_mel * T_text)
        for s in range(T_text):
            attn[t, s] = np.exp(-0.5 * ((s - center) / 1.5) ** 2)
        attn[t] /= attn[t].sum()
    attn += noise * np.random.rand(*attn.shape)
    attn /= attn.sum(axis=1, keepdims=True)
    return attn

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

attn_good = simulate_attention(T_text, T_mel, noise=0.02)
im1 = axes[0].imshow(attn_good, aspect='auto', origin='lower', cmap='Blues')
axes[0].set_title('Good Alignment: clean diagonal\n(model reads text left to right)')
axes[0].set_xlabel('Text position (characters)')
axes[0].set_ylabel('Mel frame (time)')
fig.colorbar(im1, ax=axes[0])

attn_bad = np.zeros((T_mel, T_text))
attn_bad[:, T_text // 3] = 0.8   # stuck on one character
attn_bad += 0.01
attn_bad /= attn_bad.sum(axis=1, keepdims=True)
im2 = axes[1].imshow(attn_bad, aspect='auto', origin='lower', cmap='Reds')
axes[1].set_title('Bad Alignment: attention collapse\n(stuck repeating one character)')
axes[1].set_xlabel('Text position (characters)')
axes[1].set_ylabel('Mel frame (time)')
fig.colorbar(im2, ax=axes[1])

plt.suptitle('Tacotron 2 Attention Alignment Visualization', fontsize=13)
plt.tight_layout(); plt.show()

## Inference: Text -> Mel -> Audio

```python
# Simplified inference loop
encoder_output = encoder(text_ids)          # (B, T_text, 512)
mel_frame      = torch.zeros(B, 80)         # start token (silence)
attention_weights = torch.zeros(B, T_text)

mel_outputs = []
for step in range(max_steps):
    prenet_out = prenet(mel_frame)
    context, attention_weights = attention(decoder_rnn_hidden, encoder_output, ...)
    mel_frame, stop_logit = decoder_step(prenet_out, context)
    mel_outputs.append(mel_frame)

    if torch.sigmoid(stop_logit) > 0.5:
        break

mel_spectrogram = postnet(torch.stack(mel_outputs, dim=-1))
audio           = vocoder(mel_spectrogram)   # WaveNet or HiFi-GAN
```

## Using Pre-trained Tacotron 2 with PyTorch Hub

```python
import torch

# Load from PyTorch Hub (requires internet)
tacotron2 = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub',
                            'nvidia_tacotron2', model_math='fp16')
waveglow  = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub',
                            'nvidia_waveglow',  model_math='fp16')

tacotron2 = tacotron2.to('cuda').eval()
waveglow  = waveglow.to('cuda').eval()

text   = "Hello, this is a neural text to speech system."
tokens = tacotron2.text_to_sequence(text)

with torch.no_grad():
    mel, _, _ = tacotron2.infer(tokens)
    audio     = waveglow.infer(mel)
```

## Summary

| Component | Role |
|-----------|------|
| **Embedding + Conv + BiLSTM** | Encoder: text -> context vectors |
| **Location-sensitive attention** | Align decoder frames to encoder tokens |
| **Prenet** | Always-dropout input layer for variation |
| **Decoder LSTM** | Autoregressive mel frame prediction |
| **Postnet** | Refine mel with 5-layer conv residual |
| **Stop token** | Learn when to stop generation |
| **Vocoder** | Mel -> waveform (WaveNet, WaveGlow, HiFi-GAN) |

**References:**
- Tacotron 2: [arxiv 1712.05884](https://arxiv.org/abs/1712.05884)
- HiFi-GAN vocoder: [arxiv 2010.05646](https://arxiv.org/abs/2010.05646)
- FastSpeech 2 (non-autoregressive): [arxiv 2006.04558](https://arxiv.org/abs/2006.04558)